# 04 — Observability: Telemetry, Decision Logging & Metrics

**Session: Month 6, Session 1**

## Learning Objectives
1. Apply the `@instrument` decorator to trace function execution
2. Read and interpret structured decision logs
3. Query metrics to understand agent performance
4. Understand the OpenTelemetry → Honeycomb/X-Ray pipeline

## The Observability Stack

```
Agent Function
   │
   ├─► @instrument decorator → Span (trace_id, duration, status)
   │
   ├─► DecisionLogger → JSON Lines → CloudWatch Logs
   │
   └─► AgentMetrics → counters/histograms → CloudWatch/Prometheus

All three feed into:
  - Honeycomb traces (latency breakdown)
  - CloudWatch Logs Insights (audit queries)
  - Grafana dashboards (business metrics)
```

In [ ]:
import sys, json, asyncio, time
sys.path.insert(0, '../backend')

from app.observability.telemetry import instrument, TelemetryContext
from app.observability.decision_logger import DecisionLogger, ToolCallRecord, get_decision_logger
from app.observability.metrics import AgentMetrics
from app.observability.tracer import Tracer, TraceContext

## Part 1: @instrument Decorator

In [ ]:
# Instrument a simulated tool call
@instrument(name='tool:process_refund', category='tool', log_args=False)
async def simulated_process_refund(order_id: str, amount: float):
    await asyncio.sleep(0.1)  # simulate DB call
    return {'success': True, 'refund_id': f'REF-{order_id}', 'amount': amount}

@instrument(name='tool:lookup_order', category='tool')
async def simulated_lookup_order(order_id: str):
    await asyncio.sleep(0.05)
    return {'order_id': order_id, 'status': 'processing', 'total': 125.00}

# Run them
print('Running instrumented functions...')
r1 = await simulated_lookup_order('ORD-10001')
r2 = await simulated_process_refund('ORD-10001', 50.0)
print(f'Lookup result: {r1}')
print(f'Refund result: {r2}')
print('\nSpans emitted to logs ↑ (look for SPAN | lines above)')

In [ ]:
# Use TelemetryContext as a context manager for fine-grained control
async def run_with_context():
    async with TelemetryContext('llm_call', category='llm') as ctx:
        ctx.set_attribute('model', 'llama3.2')
        ctx.set_attribute('prompt_tokens', 245)
        await asyncio.sleep(0.3)  # simulate LLM latency
        ctx.set_attribute('completion_tokens', 89)
        ctx.set_attribute('total_tokens', 334)
    
    print(f'Span completed | name={ctx.name} | duration={1000*(time.time()-ctx.start_time):.0f}ms')
    print(f'Attributes: {ctx.attributes}')

await run_with_context()

## Part 2: Decision Logging

In [ ]:
# Simulate a complete agent turn and inspect the decision log
import io
from unittest.mock import patch

logged_lines = []

def capture_log(msg):
    logged_lines.append(msg)

logger = DecisionLogger()

# Simulate an agent turn
log = logger.start_decision('conv-demo-001', 'I need a refund for order ORD-10001')
log.pii_detected = False
log.injection_detected = False
log.reflection_passed = True
log.constitutional_score = 0.95

log.add_tool_call(ToolCallRecord(
    tool_name='lookup_order',
    args={'order_id': 'ORD-10001'},
    result={'order_id': 'ORD-10001', 'status': 'delivered', 'total': 125.0},
    duration_ms=45.2,
    success=True,
    risk_score=0.0,
    policy_action='allow',
))

log.add_tool_call(ToolCallRecord(
    tool_name='process_refund',
    args={'order_id': 'ORD-10001', 'amount': 125.0, 'reason': 'Damaged item'},
    result={'success': True, 'refund_id': 'REF-ABC123', 'amount': 125.0},
    duration_ms=132.8,
    success=True,
    risk_score=0.62,
    policy_action='allow',
    hitl_approval_id=None,
))

log.final_response = 'I have processed your refund of $125.00. Refund ID: REF-ABC123.'
log.total_duration_ms = 1250.5
log.llm_model = 'llama3.2'

# Inspect the JSON before committing
parsed = json.loads(log.to_json())
print('Decision Log Structure:')
print(json.dumps(parsed, indent=2, default=str)[:2000])

In [ ]:
# CloudWatch Logs Insights query to find this log
print('CloudWatch Logs Insights queries for /agent/decisions:')
print()
print('# Find all refunds > $500 in the last 30 days:')
print("""
fields @timestamp, conversation_id, decision_id, total_duration_ms
| filter tool_calls.0.tool_name = 'process_refund'
| filter tool_calls.0.args.amount > 500
| sort @timestamp desc
| limit 50
""")

print('# Find all decisions with PII detected:')
print("""
fields @timestamp, conversation_id, decision_id
| filter pii_detected = true
| sort @timestamp desc
| limit 100
""")

print('# p99 decision latency by tool:')
print("""
stats pct(total_duration_ms, 99) as p99 by tool_calls.0.tool_name
| sort p99 desc
""")

## Part 3: Metrics

In [ ]:
import matplotlib.pyplot as plt
import random

metrics = AgentMetrics()

# Simulate 100 requests
tools = ['lookup_order', 'process_refund', 'cancel_order', 'search_documents', 'add_loyalty_points']
for _ in range(100):
    status = 'success' if random.random() > 0.05 else 'error'
    metrics.requests_total.inc(status=status)
    
    tool = random.choice(tools)
    metrics.tool_calls_total.inc(tool_name=tool, status=status)
    metrics.tool_duration.observe(random.uniform(0.05, 2.0))
    metrics.risk_score_distribution.observe(random.uniform(0, 1))

snapshot = metrics.snapshot()
print('Metrics Snapshot:')
print(json.dumps(snapshot, indent=2))

emf = metrics.emit_cloudwatch_emf()
print('\nCloudWatch EMF (auto-parsed as metrics):')
print(json.dumps(emf, indent=2))

## Lab Exercise: Add Honeycomb Exporter

1. Sign up for a free Honeycomb account at honeycomb.io
2. Get your API key and set `OTLP_ENDPOINT=https://api.honeycomb.io`
3. Set the header `x-honeycomb-team=YOUR_API_KEY`
4. Run the agent and observe traces in Honeycomb
5. Create a custom query: **p99 latency by tool_name for the last hour**

### CloudWatch Embedded Metrics (Alternative)
If you have an AWS account, the `emit_cloudwatch_emf()` JSON logged to
stdout is automatically parsed by CloudWatch as custom metrics when
running in ECS/Lambda — no SDK needed!